# Bias in Data 
"Garbage in, garbage out": a model learns whatever the data shows, including
unrepresentative sampling and unfair historical labels.

1. Synthetic population with two groups (A majority, B minority)
2. Sampling bias: B underrepresented -> worse accuracy for B
3. Selection / survivorship bias: only approved applicants have outcomes
4. Fairness metrics: demographic parity, equal opportunity, equalized odds
5. Historical (label) bias and the "remove the sensitive feature" myth (proxies)
6. Mitigation: rebalancing, reweighing (Kamiran-Calders), group thresholds
7. Trade-offs compared on one table

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(11)

## 1. Synthetic population
Each person has a latent "creditworthiness" `q`. Two observed features, `income`
and `savings`, relate to `q` *differently* per group (B's income is a noisier,
lower signal, e.g. measurement bias). `neighborhood` is a proxy for group.
True repayment label `y_true` depends only on `q` (equally for both groups).
Historical decisions `y_hist` additionally penalised group B.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

def make_population(n=20000, frac_b=0.30, hist_penalty=1.0, rng=rng):
    g = (rng.random(n) < frac_b).astype(int)                 # 1 = group B
    q = rng.normal(0, 1, n)
    income = np.where(g == 1, 0.5 * q + rng.normal(-0.4, 0.9, n), 1.0 * q + rng.normal(0, 0.5, n))
    savings = np.where(g == 1, 1.0 * q + rng.normal(0, 0.5, n), 0.4 * q + rng.normal(0, 1.0, n))
    nbhd = (rng.random(n) < np.where(g == 1, 0.85, 0.15)).astype(float)   # proxy for group
    y_true = (rng.random(n) < sigmoid(2.0 * q)).astype(int)
    y_hist = (rng.random(n) < sigmoid(2.0 * q - hist_penalty * g)).astype(int)
    return dict(g=g, q=q, X=np.c_[income, savings, nbhd], y_true=y_true, y_hist=y_hist)

pop = make_population()
print("group B share:", pop["g"].mean().round(3))
for gg, nm in [(0, "A"), (1, "B")]:
    m = pop["g"] == gg
    print(f"group {nm}: true repay rate {pop['y_true'][m].mean():.3f} | historical approval rate {pop['y_hist'][m].mean():.3f}")

## 2. Models and metrics from scratch
Logistic regression (gradient descent, supports sample weights).
* **Demographic parity difference**: `P(yhat=1|A) - P(yhat=1|B)`
* **Equal opportunity difference**: `TPR_A - TPR_B` (TPR = P(yhat=1 | y=1, group))
* **Equalized odds**: also compare FPR.

In [ ]:
def fit_logreg(X, y, w=None, lr=0.3, epochs=800, l2=1e-3):
    n, d = X.shape; w = np.ones(n) if w is None else w; w = w / w.mean()
    mu, sd = X.mean(0), X.std(0) + 1e-9; Z = (X - mu) / sd; b = np.zeros(d); b0 = 0.0
    for _ in range(epochs):
        p = sigmoid(Z @ b + b0); r = w * (p - y)
        b -= lr * (Z.T @ r / n + l2 * b); b0 -= lr * r.mean()
    return dict(b=b, b0=b0, mu=mu, sd=sd)

def predict_proba(m, X): return sigmoid(((X - m["mu"]) / m["sd"]) @ m["b"] + m["b0"])

def rates(y, yhat, g):
    out = {}
    for gg in (0, 1):
        m = g == gg; pos = m & (y == 1); neg = m & (y == 0)
        out[gg] = dict(sel=yhat[m].mean(), tpr=yhat[pos].mean(), fpr=yhat[neg].mean(), acc=(yhat[m] == y[m]).mean(), n=int(m.sum()))
    return out

def fairness_report(y, yhat, g):
    r = rates(y, yhat, g)
    return dict(acc=float((yhat == y).mean()), accA=r[0]["acc"], accB=r[1]["acc"],
                dp=r[0]["sel"] - r[1]["sel"], eo=r[0]["tpr"] - r[1]["tpr"], fpr_gap=r[0]["fpr"] - r[1]["fpr"])

def show(name, rep):
    print(f"{name:28s} acc {rep['acc']:.3f} (A {rep['accA']:.3f}, B {rep['accB']:.3f}) | DP gap {rep['dp']:+.3f} | EO gap {rep['eo']:+.3f} | FPR gap {rep['fpr_gap']:+.3f}")

# held-out test population labelled with TRUE repayment
test = make_population(n=20000, rng=np.random.default_rng(99))

## 3. Sampling bias
Train on samples where group B makes up 30% (representative), 10%, 3%. Evaluate on a
representative, truthfully labelled test set. Accuracy per group shows who pays.

In [ ]:
def subsample_groups(d, frac_b, n_total, rng):
    nb = int(n_total * frac_b); na = n_total - nb
    ia = rng.choice(np.where(d["g"] == 0)[0], na, replace=False); ib = rng.choice(np.where(d["g"] == 1)[0], nb, replace=False)
    idx = np.r_[ia, ib]; return {k: v[idx] for k, v in d.items()}

# group B reuses the same relation, but a model fit on mostly-A data fits A's relation.
# To make that bite, the model here uses only income+savings (no proxy).
feat = [0, 1]
fracs = [0.30, 0.10, 0.03, 0.01]; accs = []
for fb in fracs:
    d = subsample_groups(pop, fb, 3000, np.random.default_rng(5)); m = fit_logreg(d["X"][:, feat], d["y_true"])
    yhat = (predict_proba(m, test["X"][:, feat]) > .5).astype(int); r = rates(test["y_true"], yhat, test["g"])
    accs.append((r[0]["acc"], r[1]["acc"], r[0]["tpr"], r[1]["tpr"])); print(f"B share {fb:4.0%}: acc A {r[0]['acc']:.3f}, acc B {r[1]['acc']:.3f} | TPR A {r[0]['tpr']:.3f}, TPR B {r[1]['tpr']:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 4)); a = np.array(accs)
ax.plot(fracs, a[:, 0], 'o-', label="accuracy A"); ax.plot(fracs, a[:, 1], 's-', label="accuracy B")
ax.plot(fracs, a[:, 2], 'o--', c='C0', alpha=.5, label="TPR A"); ax.plot(fracs, a[:, 3], 's--', c='C1', alpha=.5, label="TPR B")
ax.set_xscale('log'); ax.set_xlabel("share of group B in training data"); ax.set_title("Underrepresentation hurts group B"); ax.legend()
plt.tight_layout(); plt.savefig('plots/01_sampling_bias.png', dpi=110); plt.close()

## 4. Selection (survivorship) bias
We only observe repayment for people who were *approved* in the past. A model trained
on those rows never sees how rejected applicants would have behaved, so its picture of
risk is skewed (rejected people are missing, not safe).

In [ ]:
approve_score = pop["X"][:, 0] - 0.5 * pop["g"] + rng.normal(0, 0.5, len(pop["g"]))
approved = approve_score > 0.2
print(f"Approved: {approved.mean():.1%} overall | A {approved[pop['g']==0].mean():.1%} | B {approved[pop['g']==1].mean():.1%}")
print(f"Repay rate among approved {pop['y_true'][approved].mean():.3f} vs whole population {pop['y_true'].mean():.3f}")
sel = {k: v[approved] for k, v in pop.items()}
m_sel = fit_logreg(sel["X"][:, feat], sel["y_true"]); m_full = fit_logreg(pop["X"][:, feat], pop["y_true"])
for nm, mm in [("trained on approved only", m_sel), ("trained on everyone", m_full)]:
    yhat = (predict_proba(mm, test["X"][:, feat]) > .5).astype(int); show(nm, fairness_report(test["y_true"], yhat, test["g"]))

## 5. Historical (label) bias, and why dropping the sensitive column is not enough
Train on `y_hist` (biased past decisions) and evaluate against `y_true`.
Variants: (a) with group as a feature, (b) without group but WITH the neighborhood proxy,
(c) without group and without proxy.

In [ ]:
def with_group(X, g): return np.c_[X, g]
Xtr_full = pop["X"]; gtr = pop["g"]; ytr = pop["y_hist"]
variants = {
    "(a) features+group": (with_group(Xtr_full, gtr), with_group(test["X"], test["g"])),
    "(b) no group, proxy kept": (Xtr_full, test["X"]),
    "(c) no group, no proxy": (Xtr_full[:, feat], test["X"][:, feat]),
}
reps = {}
for nm, (Xa, Xb) in variants.items():
    m = fit_logreg(Xa, ytr); yhat = (predict_proba(m, Xb) > .5).astype(int)
    reps[nm] = fairness_report(test["y_true"], yhat, test["g"]); show(nm, reps[nm])
m_fair = fit_logreg(Xtr_full[:, feat], pop["y_true"]); yhat = (predict_proba(m_fair, test["X"][:, feat]) > .5).astype(int)
show("(ref) trained on true labels", fairness_report(test["y_true"], yhat, test["g"]))
print("Correlation(neighborhood, group) =", np.corrcoef(pop["X"][:, 2], pop["g"])[0, 1].round(3))

## 6. Mitigation
**Reweighing** (Kamiran & Calders): give each (group, label) cell the weight
`w(g,y) = P(g) P(y) / P(g,y)`. After weighting, group and label are independent in the
training data, so the model can't gain by using group (or its proxies) to predict label.
Other fixes: oversampling the minority group, and per-group decision thresholds.

In [ ]:
def reweigh(g, y):
    w = np.zeros(len(y))
    for gg in (0, 1):
        for yy in (0, 1):
            cell = (g == gg) & (y == yy)
            if cell.any(): w[cell] = (g == gg).mean() * (y == yy).mean() / cell.mean()
    return w

W = reweigh(gtr, ytr)
for gg in (0, 1):
    for yy in (0, 1):
        c = (gtr == gg) & (ytr == yy); print(f"group {'AB'[gg]} label {yy}: n={c.sum():5d} weight={W[c][0]:.3f}")
print("weighted positive rate A vs B:", [round(float(np.average(ytr[gtr == k], weights=W[gtr == k])), 3) for k in (0, 1)])

def oversample_minority(X, y, g, rng):
    ib = np.where(g == 1)[0]; ia = np.where(g == 0)[0]; extra = rng.choice(ib, len(ia) - len(ib), replace=True)
    idx = np.r_[np.arange(len(g)), extra]; return X[idx], y[idx], g[idx]

def group_thresholds(p, y, g, target_tpr):
    """Pick per-group thresholds to hit the same TPR on labelled data (equal opportunity post-processing)."""
    th = {}
    for gg in (0, 1):
        s = np.sort(p[(g == gg) & (y == 1)]); th[gg] = np.quantile(s, 1 - target_tpr)
    return th

results = {}
Xa, Xb = variants["(b) no group, proxy kept"]
m0 = fit_logreg(Xa, ytr); p0 = predict_proba(m0, Xb); results["baseline (biased labels)"] = fairness_report(test["y_true"], (p0 > .5).astype(int), test["g"])
mw = fit_logreg(Xa, ytr, w=W); pw = predict_proba(mw, Xb); results["reweighing"] = fairness_report(test["y_true"], (pw > .5).astype(int), test["g"])
Xo, yo, go = oversample_minority(Xa, ytr, gtr, np.random.default_rng(2)); mo = fit_logreg(Xo, yo)
results["oversample group B"] = fairness_report(test["y_true"], (predict_proba(mo, Xb) > .5).astype(int), test["g"])
# group thresholds on baseline, tuned against true-repayment labels from a separate calibration slice
cal = make_population(n=10000, rng=np.random.default_rng(123)); pc = predict_proba(m0, cal["X"])
th = group_thresholds(pc, cal["y_true"], cal["g"], target_tpr=0.80)
yhat_gt = (p0 > np.where(test["g"] == 1, th[1], th[0])).astype(int)
results["group thresholds (EO)"] = fairness_report(test["y_true"], yhat_gt, test["g"])
mt = fit_logreg(Xtr_full[:, feat], pop["y_true"]); results["oracle: true labels, no proxy"] = fairness_report(test["y_true"], (predict_proba(mt, test["X"][:, feat]) > .5).astype(int), test["g"])
print()
for k, v in results.items(): show(k, v)
print("thresholds used:", {('A', 'B')[k]: round(float(v), 3) for k, v in th.items()})

## 7. Trade-offs plot
Fairness gaps (closer to 0 is fairer) and overall accuracy against the truth.

In [ ]:
names = list(results); xi = np.arange(len(names))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(xi - .2, [results[k]["dp"] for k in names], .4, label="demographic parity gap")
ax[0].bar(xi + .2, [results[k]["eo"] for k in names], .4, label="equal opportunity gap")
ax[0].axhline(0, c='k', lw=.8); ax[0].set_xticks(xi); ax[0].set_xticklabels(names, rotation=25, ha='right'); ax[0].legend(); ax[0].set_title("Fairness gaps (A minus B)")
ax[1].bar(xi - .2, [results[k]["accA"] for k in names], .4, label="accuracy A"); ax[1].bar(xi + .2, [results[k]["accB"] for k in names], .4, label="accuracy B")
ax[1].set_ylim(.5, 1); ax[1].set_xticks(xi); ax[1].set_xticklabels(names, rotation=25, ha='right'); ax[1].legend(); ax[1].set_title("Accuracy vs true repayment")
plt.tight_layout(); plt.savefig('plots/02_mitigation.png', dpi=110); plt.close()

Distribution view: model score by group before and after reweighing.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, (nm, p) in zip(ax, [("baseline", p0), ("reweighing", pw)]):
    for gg, c in [(0, 'C0'), (1, 'C1')]: a.hist(p[test["g"] == gg], 40, alpha=.5, density=True, color=c, label="AB"[gg])
    a.set_title(f"Score distribution: {nm}"); a.set_xlabel("P(approve)"); a.legend()
plt.tight_layout(); plt.savefig('plots/03_score_distributions.png', dpi=110); plt.close()

**Takeaways**: (1) collect representative data, (2) audit metrics per group, not just overall
accuracy, (3) removing the sensitive column leaves proxies, (4) fairness criteria can conflict
(demographic parity vs equal opportunity vs accuracy), so pick based on the application and
keep a human in the loop. Golden rule: audit the data before trusting the model.

In [ ]:
# Optional sanity check of the logistic regression against sklearn
try:
    from sklearn.linear_model import LogisticRegression
    Xs = pop["X"][:, feat]; mm = fit_logreg(Xs, pop["y_true"])
    sk = LogisticRegression(C=1e3).fit((Xs - Xs.mean(0)) / Xs.std(0), pop["y_true"])
    print("agreement with sklearn predictions:", np.mean(sk.predict((Xs - Xs.mean(0)) / Xs.std(0)) == (predict_proba(mm, Xs) > .5)).round(4))
except Exception as e:
    print("sklearn check skipped:", e)